# ICD Code Mapping: Version Inference, GEM, and CCS

Claims databases (MarketScan, LCED) carry ICD diagnosis codes in two versions:

- **ICD-9-CM** — used in the US until September 2015
- **ICD-10-CM** — mandatory from October 2015 onwards

This notebook covers:
1. Version inference from code structure
2. CMS General Equivalence Mappings (GEM) — ICD-9 ↔ ICD-10 translation
3. AHRQ Clinical Classifications Software (CCS) — grouping codes into clinical categories
4. Building disease phenotype code lists

All patterns translate the original R `diagnosis code.R` workflow.

In [1]:
import re
import pandas as pd

## 1. ICD Version Structure

ICD-9 and ICD-10 codes have distinct syntactic signatures:

| Version | Pattern | Examples |
|---|---|---|
| ICD-9-CM | 3–5 digits, optionally dotted; E/V prefix codes | `250.00`, `E11`, `V45.86` |
| ICD-10-CM | Letter + 2 digits + optional 1–4 chars | `E11.9`, `I25.10`, `Z79.4` |

The `dxver` column in MarketScan / LCED records this as `'9'` or `'0'`, but is frequently null for ICD-9 E/V codes.

In [2]:
# Representative diagnosis codes from LCED / MarketScan
test_codes = pd.DataFrame(
    {
        "dx": ["250.00", "E11.9", "I25.10", "410.00", "V45.86", "428.0", "Z79.4", "160.1", "E11", "25000"],
        "dxver": ["9", "0", "0", None, None, None, "0", None, None, "9"],
    }
)
test_codes

,dx,dxver
0,250.00,9
1,E11.9,0
2,I25.10,0
3,410.00,None
4,V45.86,None
5,428.0,None
6,Z79.4,0
7,160.1,None
8,E11,None
9,25000,9


## 2. Version Inference

`normalize.infer_icd_version` fills in null `dxver` values by inspecting the code pattern. ICD-9 E/V prefix codes — `E\d+` and `V\d+` — are the main source of missingness in MarketScan.

In [3]:
_ICD9_PATTERN = re.compile(r"^[VvEe]?\d")
_ICD10_PATTERN = re.compile(r"^[A-Za-z]\d")


def infer_icd_version(dx, dxver):
    """Return '9' or '0'; fall back to regex if dxver is null."""
    if pd.notna(dxver) and dxver in ("9", "0"):
        return dxver
    if pd.isna(dx):
        return None
    code = str(dx).strip()
    if _ICD9_PATTERN.match(code):
        return "9"
    if _ICD10_PATTERN.match(code):
        return "0"
    return None


test_codes["dxver_inferred"] = [infer_icd_version(row.dx, row.dxver) for _, row in test_codes.iterrows()]
test_codes

,dx,dxver,dxver_inferred
0,250.00,9,9
1,E11.9,0,0
2,I25.10,0,0
3,410.00,None,9
4,V45.86,None,9
5,428.0,None,9
6,Z79.4,0,0
7,160.1,None,9
8,E11,None,9
9,25000,9,9


## 3. CMS General Equivalence Mappings (GEM)

The CMS GEM files provide bidirectional approximate translations between ICD-9 and ICD-10. `ehrdata.io.source.vocab.icd.load_gem` is a stub that will load the CMS file once provided.

```
2018_I10gem.txt  — ICD-10-CM → ICD-9-CM
2018_I9gem.txt   — ICD-9-CM → ICD-10-CM
```

The original R pipeline used the `2018_I10gem.txt` to check which phenotype codes had equivalents in the data:
```r
icd.gem <- fread(".../2018_I10gem.txt", col.names=c("icd10", "icd9"))
sum(diagnosis.df$dx %in% icd.gem$icd10)
```

Below we show the equivalent pattern with a small inline GEM table.

In [4]:
# Minimal GEM table (ICD-10 → ICD-9)
gem_i10 = pd.DataFrame(
    {
        "icd10": ["E119", "E11", "I2510", "I110", "I259", "I509", "G309"],
        "icd9": ["25000", "250", "41401", "40210", "41400", "42800", "33100"],
        "flags": ["10000", "10000", "10000", "10000", "10000", "10000", "10000"],
    }
)

# Strip dots from codes before lookup (LCED stores codes without dots)
test_codes["dx_nodot"] = test_codes["dx"].str.replace(".", "", regex=False)

# ICD-10 codes with a GEM equivalent
mapped = test_codes[test_codes["dx_nodot"].isin(gem_i10["icd10"])]
print(f"Codes with GEM equivalent: {len(mapped)} / {len(test_codes)}")
print()

# Look up the ICD-9 equivalent
result = mapped.merge(gem_i10[["icd10", "icd9"]], left_on="dx_nodot", right_on="icd10", how="left")
result[["dx", "dxver_inferred", "icd9"]]

Codes with GEM equivalent: 3 / 10



,dx,dxver_inferred,icd9
0,E11.9,0,25000
1,I25.10,0,41401
2,E11,9,250


## 4. AHRQ Clinical Classifications Software (CCS)

CCS groups the ~70,000 ICD codes into ~500 clinically meaningful categories. This is useful for exploratory analyses (e.g. "all cardiovascular disease") without enumerating every ICD code.

`ehrdata.io.source.vocab.icd.load_ccs_map` is a stub that will load the AHRQ CCS CSV once provided.

### ICD-10-CM CCS category search (R pattern → Python)

```r
# R:
grep("heart failure", icd.ccs.disease, ignore.case=TRUE, value=TRUE)
```
```python
# Python equivalent:
ccs_df["category_desc"].str.contains("heart failure", case=False)
```

In [5]:
# Minimal ICD-10-CM CCS table (mirrors DXCCSR_v2020-3.CSV structure)
ccs_icd10 = pd.DataFrame(
    {
        "icd10": ["E119", "E118", "I509", "I2510", "I110", "G309", "N183", "J449", "C50.9"],
        "icd10_desc": [
            "T2DM unspecified",
            "T2DM other",
            "Heart failure",
            "Coronary artery disease",
            "Hypertensive HF",
            "Alzheimer disease",
            "CKD stage 3",
            "COPD unspecified",
            "Breast cancer",
        ],
        "ccs_category": ["END002", "END002", "CIR019", "CIR007", "CIR007", "NVS010", "GEN003", "RSP010", "NEO011"],
        "ccs_desc": [
            "Diabetes mellitus without complication",
            "Diabetes mellitus without complication",
            "Heart failure",
            "Coronary atherosclerosis and other heart disease",
            "Coronary atherosclerosis and other heart disease",
            "Alzheimer disease and related disorders",
            "Chronic kidney disease",
            "Chronic obstructive pulmonary disease and bronchiectasis",
            "Cancer of breast",
        ],
    }
)


def search_ccs(df, pattern, col="ccs_desc"):
    mask = df[col].str.contains(pattern, case=False, na=False)
    return df[mask][["ccs_category", "ccs_desc"]].drop_duplicates()


for term in ["heart failure", "diabetes", "chronic kidney", "cancer", "coronary"]:
    hits = search_ccs(ccs_icd10, term)
    print(f"'{term}': {hits['ccs_category'].tolist()}")

'heart failure': ['CIR019']
'diabetes': ['END002']
'chronic kidney': ['GEN003']
'cancer': ['NEO011']
'coronary': ['CIR007']


## 5. Building Disease Phenotype Code Lists

The original R pipeline defined disease cohorts using explicit ICD code lists. Here we translate the most common ones — Type 2 Diabetes, AMI, Stroke, CHF — into Python dictionaries.

In [6]:
# ICD-9 codes for Type 2 Diabetes (from lced_template_analysis.R)
T2D_ICD9 = [
    "250.0",
    "25000",
    "25002",
    "2501",
    "25010",
    "25012",
    "2504",
    "25040",
    "25042",
    "2505",
    "25050",
    "25052",
    "2506",
    "25060",
    "25062",
    "2507",
    "25070",
    "25072",
    "2509",
    "25090",
    "25092",
]
# ICD-10 prefix
T2D_ICD10_PREFIX = "E11"

# AMI (Acute Myocardial Infarction)
AMI_ICD9 = ["410"] + [f"41{x}" for x in ["00", "01", "02", "03", "04", "05", "06", "07", "08", "09"]]
AMI_ICD10_PREFIX = "I21"

# Stroke
STROKE_ICD9_PREFIXES = ["160", "161", "162", "163", "430", "431", "432", "433", "434", "435"]
STROKE_ICD10_PREFIXES = ["I60", "I61", "I62", "I63", "I64"]

# CHF (Congestive Heart Failure)
CHF_ICD9 = ["4280", "428"] + [f"428{x}" for x in range(10)]
CHF_ICD10_PREFIX = "I50"

print("T2D ICD-9 codes:", T2D_ICD9[:6], "...")
print("T2D ICD-10 prefix:", T2D_ICD10_PREFIX + "*")

T2D ICD-9 codes: ['250.0', '25000', '25002', '2501', '25010', '25012'] ...
T2D ICD-10 prefix: E11*


## 6. Phenotype Matching in a Diagnosis Table

Once you have a canonical `diagnosis` DataFrame from the LCED adapter, matching against these code lists is straightforward.

In [7]:
# Synthetic diagnosis table (mirrors lced.build_diagnosis output)
diagnosis = pd.DataFrame(
    {
        "patient_id": ["P001", "P001", "P002", "P003", "P003", "P004", "P004"],
        "eventdate": pd.to_datetime(
            [
                "2014-03-01",
                "2015-11-15",
                "2016-07-20",
                "2013-09-05",
                "2017-04-10",
                "2014-12-01",
                "2018-02-28",
            ]
        ),
        "dx": ["E119", "I509", "25000", "I2510", "41001", "410", "E119"],
        "dxver": ["0", "0", "9", "0", "9", "9", "0"],
    }
)


def has_icd(dx_series, icd9_list=None, icd10_prefixes=None, icd9_prefixes=None):
    """Return a boolean mask for rows matching ICD-9 list or ICD-10 prefix."""
    mask = pd.Series(False, index=dx_series.index)
    if icd9_list:
        dx_nodot = dx_series.str.replace(".", "", regex=False)
        mask |= dx_nodot.isin(icd9_list)
    if icd10_prefixes:
        if isinstance(icd10_prefixes, str):
            icd10_prefixes = [icd10_prefixes]
        for pfx in icd10_prefixes:
            mask |= dx_series.str.startswith(pfx, na=False)
    if icd9_prefixes:
        for pfx in icd9_prefixes:
            mask |= dx_series.str.startswith(pfx, na=False)
    return mask


diagnosis["t2d"] = has_icd(diagnosis["dx"], icd9_list=T2D_ICD9, icd10_prefixes="E11")
diagnosis["ami"] = has_icd(diagnosis["dx"], icd9_list=AMI_ICD9, icd10_prefixes="I21")
diagnosis["stroke"] = has_icd(diagnosis["dx"], icd9_prefixes=STROKE_ICD9_PREFIXES, icd10_prefixes=STROKE_ICD10_PREFIXES)
diagnosis["chf"] = has_icd(diagnosis["dx"], icd9_list=CHF_ICD9, icd10_prefixes="I50")

print(diagnosis.to_string(index=False))
print()
print("Patients with T2D:")
print(diagnosis[diagnosis["t2d"]]["patient_id"].unique())

patient_id  eventdate    dx dxver   t2d   ami  stroke   chf
      P001 2014-03-01  E119     0  True False   False False
      P001 2015-11-15  I509     0 False False   False  True
      P002 2016-07-20 25000     9  True False   False False
      P003 2013-09-05 I2510     0 False False   False False
      P003 2017-04-10 41001     9 False False   False False
      P004 2014-12-01   410     9 False  True   False False
      P004 2018-02-28  E119     0  True False   False False

Patients with T2D:
['P001' 'P002' 'P004']


## 7. Cross-Version Cohort Definition

Studies spanning the ICD-9→ICD-10 transition (October 2015) need to include both code versions. The canonical pattern:

In [8]:
# Patients with ≥1 T2D code in any version
t2d_patients = (
    diagnosis[diagnosis["t2d"]]
    .groupby("patient_id")["eventdate"]
    .min()  # index date = first T2D code
    .rename("t2d_index_date")
    .reset_index()
)
print("T2D cohort with index dates:")
print(t2d_patients)

# Verify code versions present
print()
print("ICD versions in T2D rows:")
print(diagnosis[diagnosis["t2d"]][["dx", "dxver"]].value_counts().to_string())

T2D cohort with index dates:
  patient_id t2d_index_date
0       P001     2014-03-01
1       P002     2016-07-20
2       P004     2018-02-28

ICD versions in T2D rows:
dx     dxver
E119   0        2
25000  9        1


## 8. Using the `vocab/icd` Stubs\n\nThe `ehrdata.io.source.vocab.icd` module provides two stub functions that will load external mapping files when available:\n\n```python\nfrom ehrdata.io.source.vocab import icd\n\n# When CMS GEM file is available:\ngem = icd.load_gem_map(\"2018_I10gem.txt\")      # columns: source_code, target_code, flags\n\n# When AHRQ CCS file is available:\nccs = icd.load_ccs_map(\"DXCCSR_v2020-3.CSV\")  # columns: icd_code, ccs_category, ccs_description\n```\n\nUntil those files are provided, use the pattern from Section 5 above (explicit code lists) which is fully reproducible without external dependencies.

In [9]:
from ehrdata.io.source.vocab import icd

# These will raise NotImplementedError until the external files are supplied
try:
    gem = icd.load_gem_map("2018_I10gem.txt")
except (NotImplementedError, FileNotFoundError) as e:
    print(f"load_gem_map: {e}")

try:
    ccs = icd.load_ccs_map("DXCCSR_v2020-3.CSV")
except (NotImplementedError, FileNotFoundError) as e:
    print(f"load_ccs_map: {e}")

load_gem_map: GEM loader requires external CMS GEM file; not yet implemented.
load_ccs_map: CCS loader requires external AHRQ CCS file; not yet implemented.


## Summary

| Task | Approach |
|---|---|
| Version inference | `normalize.infer_icd_version` or regex on code structure |
| ICD-10 → ICD-9 translation | `icd.load_gem(path)` (requires CMS GEM file) |
| Clinical grouping | `icd.load_ccs_map(path)` (requires AHRQ CCS file) |
| Phenotype matching | `dx.isin(code_list)` or `dx.str.startswith(prefix)` |
| Cross-version cohort | combine ICD-9 list + ICD-10 prefix in one boolean mask |

**Next steps**: See `source_lced_cohort.ipynb` for a full T2D second-line therapy cohort that applies these phenotype definitions.